# 👗 Fashion MNIST — Multi-Class CNN Classifier
### Target: 95%+ Accuracy with Data Augmentation, BatchNorm & EarlyStopping

---

## Project Overview
Classify 10 categories of clothing from the Fashion MNIST dataset using a deep **Convolutional Neural Network (CNN)**.  
This project uses modern deep learning techniques including **data augmentation**, **batch normalization**, **dropout**, and **learning rate scheduling** to achieve 95%+ test accuracy.

**Dataset:** Fashion MNIST (70,000 grayscale 28×28 images, 10 classes)  
**Task:** Multi-class Image Classification  
**Architecture:** 3-block CNN with BatchNorm + Global Average Pooling  

| Class | Label | Class | Label |
|-------|----------|-------|----------|
| T-shirt/top | 0 | Sandal | 5 |
| Trouser | 1 | Shirt | 6 |
| Pullover | 2 | Sneaker | 7 |
| Dress | 3 | Bag | 8 |
| Coat | 4 | Ankle boot | 9 |

---

## Table of Contents
1. [Import Libraries](#1)
2. [Load & Inspect Data](#2)
3. [Normalize & Reshape](#3)
4. [Data Augmentation Pipeline](#4)
5. [Build CNN Model](#5)
6. [Callbacks](#6)
7. [Train the Model](#7)
8. [Evaluate & Plot Results](#8)
9. [Confusion Matrix](#9)
10. [Predict on Sample Images](#10)

## 1. Import Libraries

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

# Class names for Fashion MNIST
CLASS_NAMES = ['T-shirt','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']

## 2. Load & Inspect Data

In [ ]:
(train_data, train_labels), (test_data, test_labels) = keras.datasets.fashion_mnist.load_data()

print('Train data shape:', train_data.shape)   # (60000, 28, 28)
print('Test data shape: ', test_data.shape)    # (10000, 28, 28)
print('Pixel value range:', train_data.min(), '-', train_data.max())

# Preview sample images
plt.figure(figsize=(12, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(train_data[i], cmap='gray')
    plt.title(CLASS_NAMES[train_labels[i]], fontsize=8)
    plt.axis('off')
plt.suptitle('Sample Fashion MNIST Images', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Normalize & Reshape

In [ ]:
# Normalize pixel values from [0-255] to [0-1]
train_data = train_data.astype('float32') / 255.0
test_data  = test_data.astype('float32')  / 255.0

# Add channel dimension: (28,28) -> (28,28,1) for CNN
train_data = train_data[..., np.newaxis]
test_data  = test_data[..., np.newaxis]

print('Train data shape after reshape:', train_data.shape)  # (60000, 28, 28, 1)
print('Pixel range: min=', train_data.min(), ' max=', train_data.max())

## 4. Data Augmentation Pipeline

In [ ]:
BATCH_SIZE = 64
AUTOTUNE   = tf.data.AUTOTUNE

# Augmentation layer (applied only during training)
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name='augmentation')

# Build tf.data pipelines for performance
train_dataset = tf.data.Dataset.from_tensor_slices((train_data, train_labels))
train_dataset = (train_dataset
                 .shuffle(10000)
                 .batch(BATCH_SIZE)
                 .map(lambda x, y: (data_augmentation(x, training=True), y),
                      num_parallel_calls=AUTOTUNE)
                 .prefetch(AUTOTUNE))

test_dataset = tf.data.Dataset.from_tensor_slices((test_data, test_labels))
test_dataset = (test_dataset
                .batch(BATCH_SIZE)
                .prefetch(AUTOTUNE))

print('Data pipeline ready! Batches per epoch:', len(train_dataset))

## 5. Build CNN Model

In [ ]:
def build_model(input_shape=(28, 28, 1), num_classes=10):
    inputs = keras.Input(shape=input_shape)

    # ── Block 1: 32 filters ───────────────────────────────────
    x = layers.Conv2D(32, (3,3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)

    # ── Block 2: 64 filters ───────────────────────────────────
    x = layers.Conv2D(64, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)

    # ── Block 3: 128 filters ──────────────────────────────────
    x = layers.Conv2D(128, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.25)(x)

    # ── Classifier head ───────────────────────────────────────
    x = layers.GlobalAveragePooling2D()(x)   # replaces Flatten — fewer params
    x = layers.Dense(256)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs, name='FashionCNN')

model = build_model()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 6. Callbacks

In [ ]:
callbacks = [
    # Stop training if val_accuracy doesn't improve for 5 epochs
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    # Halve learning rate when stuck
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    # Save best model checkpoint
    keras.callbacks.ModelCheckpoint(
        'best_fashion_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print('Callbacks configured!')

## 7. Train the Model

In [ ]:
history = model.fit(
    train_dataset,
    epochs=50,
    validation_data=test_dataset,
    callbacks=callbacks,
    verbose=1
)

print('Training complete!')

## 8. Evaluate & Plot Results

In [ ]:
test_loss, test_acc = model.evaluate(test_dataset, verbose=0)
print(f'Test accuracy: {test_acc*100:.2f}%')
print(f'Test loss:     {test_loss:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Confusion Matrix

In [ ]:
y_pred_probs = model.predict(test_dataset)
y_pred       = np.argmax(y_pred_probs, axis=1)

print(classification_report(test_labels, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(test_labels, y_pred)
plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix — Fashion MNIST')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 10. Predict on Sample Images

In [ ]:
num_samples = 10
indices     = np.random.choice(len(test_data), num_samples, replace=False)

plt.figure(figsize=(15, 4))
for i, idx in enumerate(indices):
    img  = test_data[idx]
    true = CLASS_NAMES[test_labels[idx]]
    pred = CLASS_NAMES[np.argmax(model.predict(img[np.newaxis,...], verbose=0))]
    color = 'green' if true == pred else 'red'

    plt.subplot(2, 5, i+1)
    plt.imshow(img.squeeze(), cmap='gray')
    plt.title(f'True: {true}\nPred: {pred}', fontsize=7, color=color)
    plt.axis('off')

plt.suptitle('Green = Correct   |   Red = Wrong', fontsize=11)
plt.tight_layout()
plt.show()